# Bronze Discovery Notebook

Looking at `qec_syndromes`, `google_qec`, `qasmbench` before writing the pipeline. Trying to answer the "Questions to investigate" from `assignment/data-sources.md`.

## What's actually in Bronze

Sanity check that MinIO is reachable and see the sizes of the raw objects.

In [14]:
import io, zipfile, hashlib, re, ast
import pandas as pd

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory
from quantum_lake_student.formats import iter_b8_records, parse_01_records, b8_record_bytes

settings = Settings.from_environment()
client = minio_client(settings)

for name, size in bronze_inventory(settings):
    print(f"{size:>12,} bytes  {name}")

  14,638,673 bytes  bronze/source=google_qec/google-surface-code-curated.zip
     144,172 bytes  bronze/source=qasmbench/qasmbench-qec.zip
     358,017 bytes  bronze/source=qec_syndromes/syndromes_dataset.zip


## Pull the zips into memory

Work with in-memory than unzip to disk. Member counts are given below as well.

In [2]:
SYN_OBJ = "bronze/source=qec_syndromes/syndromes_dataset.zip"
GOOGLE_OBJ = "bronze/source=google_qec/google-surface-code-curated.zip"
QASM_OBJ = "bronze/source=qasmbench/qasmbench-qec.zip"

def get_zip_bytes(object_name: str) -> bytes:
    return client.get_object(settings.s3_bucket, object_name).read()

SYN_ZIP_BYTES = get_zip_bytes(SYN_OBJ)
GOOGLE_ZIP_BYTES = get_zip_bytes(GOOGLE_OBJ)
QASM_ZIP_BYTES = get_zip_bytes(QASM_OBJ)

def zip_names(zip_bytes: bytes):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        return z.namelist()

def zip_read(zip_bytes: bytes, member: str) -> bytes:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        return z.read(member)

print(len(zip_names(SYN_ZIP_BYTES)), "syndrome members")
print(len(zip_names(GOOGLE_ZIP_BYTES)), "google members")
print(len(zip_names(QASM_ZIP_BYTES)), "qasmbench members")

8 syndrome members
76 google members
19 qasmbench members


## Syndrome CSVs

Checking each of the 7 files for: header shape, that syndromes are 4x4 binary, whether `quantity` sums match the `nb-10M` in the filename, and whether `syndromes` alone is a safe key or if I need `(syndromes, labels)`.

In [3]:
syn_members = [n for n in zip_names(SYN_ZIP_BYTES) if n.endswith(".csv")]
print(syn_members)

def load_syndrome_csv(member):
    df = pd.read_csv(io.BytesIO(zip_read(SYN_ZIP_BYTES, member)))
    assert list(df.columns) == ["labels", "syndromes", "quantity"], df.columns
    return df

results = []
for member in syn_members:
    df = load_syndrome_csv(member)
    df["parsed"] = df["syndromes"].apply(ast.literal_eval)

    bad_shape = df["parsed"].apply(lambda s: len(s) != 4 or any(len(r) != 4 for r in s))
    bad_domain = df["parsed"].apply(lambda s: any(v not in (0, 1) for r in s for v in r))
    assert not bad_shape.any(), f"{member}: bad shape rows {bad_shape.sum()}"
    assert not bad_domain.any(), f"{member}: non-binary values {bad_domain.sum()}"

    m = re.match(r"d-(\d+)_pfr-([\d.]+)_nb-(\d+)M\.csv", member.split("/")[-1])
    d, pfr, nb_millions = int(m[1]), float(m[2]), int(m[3])
    expected_total = nb_millions * 1_000_000
    actual_total = df["quantity"].sum()

    dup_labels = df.groupby("syndromes")["labels"].nunique()
    ambiguous = (dup_labels > 1).sum()

    key_syndrome_only_unique = not df["syndromes"].duplicated().any()
    key_syn_label_unique = not df.duplicated(subset=["syndromes", "labels"]).any()

    results.append(dict(
        file=member, distance=d, pfr=pfr,
        expected_total=expected_total, actual_total=actual_total,
        matches=expected_total == actual_total,
        ambiguous_syndromes=ambiguous,
        syndrome_alone_unique=key_syndrome_only_unique,
        syndrome_plus_label_unique=key_syn_label_unique,
    ))

summary = pd.DataFrame(results)
summary


['d-3_pfr-0.000010_nb-10M.csv', 'd-3_pfr-0.000050_nb-10M.csv', 'd-3_pfr-0.000100_nb-10M.csv', 'd-3_pfr-0.000500_nb-10M.csv', 'd-3_pfr-0.001000_nb-10M.csv', 'd-3_pfr-0.005000_nb-10M.csv', 'd-3_pfr-0.010000_nb-10M.csv']


,file,distance,pfr,expected_total,actual_total,matches,ambiguous_syndromes,syndrome_alone_unique,syndrome_plus_label_unique
0,d-3_pfr-0.000010_nb-10M.csv,3,0.00001,10000000,10000000,True,0,True,True
1,d-3_pfr-0.000050_nb-10M.csv,3,0.00005,10000000,10000000,True,13,False,True
2,d-3_pfr-0.000100_nb-10M.csv,3,0.00010,10000000,10000000,True,46,False,True
3,d-3_pfr-0.000500_nb-10M.csv,3,0.00050,10000000,10000000,True,179,False,True
4,d-3_pfr-0.001000_nb-10M.csv,3,0.00100,10000000,10000000,True,467,False,True
5,d-3_pfr-0.005000_nb-10M.csv,3,0.00500,10000000,10000000,True,6244,False,True
6,d-3_pfr-0.010000_nb-10M.csv,3,0.01000,10000000,10000000,True,18210,False,True


## Does `quantity` actually matter for aggregates?

Quick check on the highest-pfr file: plain mean of `labels` vs. `quantity`-weighted mean.

In [4]:
df = load_syndrome_csv(syn_members[-1])  # highest pfr file, most error signal
unweighted_rate = df["labels"].mean()
weighted_rate = (df["labels"] * df["quantity"]).sum() / df["quantity"].sum()
print(f"unweighted (wrong): {unweighted_rate:.6f}  weighted (correct): {weighted_rate:.6f}")


unweighted (wrong): 0.461007  weighted (correct): 0.186527


## Splitting train/test by file, not by row

`pfr` lives at the file level, not per row.

In [ ]:
print(summary[["file", "pfr"]])
# so a row-level split would leak the noise regime across train/test

## Google experiments: does the directory name match `properties.yml`?

Also checking each experiment dir has all the companion files it should.

In [6]:
def parse_properties_yml(text: str) -> dict:
    out = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, _, value = line.partition(":")
        value = value.strip()
        out[key.strip()] = int(value) if value.lstrip("-").isdigit() else value
    return out

google_names = zip_names(GOOGLE_ZIP_BYTES)
experiment_dirs = sorted({n.split("/")[0] for n in google_names if "/" in n})
print(experiment_dirs)

EXPECTED_FILES = {
    "properties.yml", "layout.svg", "circuit_ideal.stim", "circuit_noisy.stim",
    "circuit_detector_error_model.dem", "pij_from_even_for_odd.dem",
    "pij_from_odd_for_even.dem", "measurements.b8", "sweep.b8",
    "detection_events.b8", "obs_flips_actual.01",
    "obs_flips_predicted_by_pymatching.01",
    "obs_flips_predicted_by_correlated_matching.01",
    "obs_flips_predicted_by_belief_matching.01",
    "obs_flips_predicted_by_tensor_network_contraction.01",
}

NAME_RE = re.compile(
    r"surface_code_b(?P<basis>[XZ])_d(?P<distance>\d+)_r(?P<rounds>\d+)"
    r"_center_(?P<row>\d+)_(?P<col>\d+)"
)

props_by_dir = {}
for d in experiment_dirs:
    present = {n[len(d) + 1:] for n in google_names if n.startswith(d + "/")}
    missing = EXPECTED_FILES - present
    props = parse_properties_yml(zip_read(GOOGLE_ZIP_BYTES, f"{d}/properties.yml").decode())
    props_by_dir[d] = props

    m = NAME_RE.match(d)
    name_vs_props = dict(
        basis=(m["basis"] == props["basis"]),
        distance=(int(m["distance"]) == props["distance"]),
        rounds=(int(m["rounds"]) == props["rounds"]),
        center_row=(int(m["row"]) == props["center_data_qubit_row"]),
        center_col=(int(m["col"]) == props["center_data_qubit_col"]),
    )

    print(d)
    print("  missing files:", missing or "none")
    print("  name vs properties.yml agree:", name_vs_props)


['surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5']
surface_code_bX_d3_r25_center_3_5
  missing files: none
  name vs properties.yml agree: {'basis': True, 'distance': True, 'rounds': True, 'center_row': True, 'center_col': True}
surface_code_bX_d3_r25_center_5_3
  missing files: none
  name vs properties.yml agree: {'basis': True, 'distance': True, 'rounds': True, 'center_row': True, 'center_col': True}
surface_code_bX_d3_r25_center_5_7
  missing files: none
  name vs properties.yml agree: {'basis': True, 'distance': True, 'rounds': True, 'center_row': True, 'center_col': True}
surface_code_bX_d3_r25_center_7_5
  missing files: none
  name vs properties.yml agree: {'basis': True, 'distance': True, 'rounds': True, 'center_row': True, 'center_col': True}
surface_code_bX_d5_r25_center_5_5
  missing files: none
  name vs properties.yml agree: {'basis': True

## `.b8` file sizes

These are Stim's bit-packed format. Using the supplied `formats.py` helpers to compute expected size from `shots * bits_per_shot` (pulled from `properties.yml`) and comparing against the real file.

In [7]:
for d in experiment_dirs:
    props = props_by_dir[d]
    shots = props["shots"]
    print(d)
    for fname, bits_key in [
        ("measurements.b8", "circuit_measurements"),
        ("sweep.b8", "circuit_sweep_bits"),
        ("detection_events.b8", "circuit_detectors"),
    ]:
        data = zip_read(GOOGLE_ZIP_BYTES, f"{d}/{fname}")
        bits = props[bits_key]
        expected_len = shots * b8_record_bytes(bits)
        try:
            records = list(iter_b8_records(data, bits_per_record=bits))
            ok = len(data) == expected_len and len(records) == shots
            print(f"  {fname}: len={len(data)} expected={expected_len} "
                  f"records={len(records)} shots={shots} -> {'OK' if ok else 'MISMATCH'}")
        except ValueError as e:
            print(f"  {fname}: FAIL ({e}); actual_len={len(data)} expected_len={expected_len}")


surface_code_bX_d3_r25_center_3_5
  measurements.b8: len=1350000 expected=1350000 records=50000 shots=50000 -> OK
  sweep.b8: len=100000 expected=100000 records=50000 shots=50000 -> OK
  detection_events.b8: len=1250000 expected=1250000 records=50000 shots=50000 -> OK
surface_code_bX_d3_r25_center_5_3
  measurements.b8: len=1350000 expected=1350000 records=50000 shots=50000 -> OK
  sweep.b8: len=100000 expected=100000 records=50000 shots=50000 -> OK
  detection_events.b8: len=1250000 expected=1250000 records=50000 shots=50000 -> OK
surface_code_bX_d3_r25_center_5_7
  measurements.b8: len=1350000 expected=1350000 records=50000 shots=50000 -> OK
  sweep.b8: len=100000 expected=100000 records=50000 shots=50000 -> OK
  detection_events.b8: len=1250000 expected=1250000 records=50000 shots=50000 -> OK
surface_code_bX_d3_r25_center_7_5
  measurements.b8: len=1350000 expected=1350000 records=50000 shots=50000 -> OK
  sweep.b8: len=100000 expected=100000 records=50000 shots=50000 -> OK
  detect

## `.01` files

Should be one 0/1 line per shot.

In [8]:
for d in experiment_dirs:
    props = props_by_dir[d]
    shots = props["shots"]
    print(d)
    for fname in sorted(n for n in EXPECTED_FILES if n.endswith(".01")):
        data = zip_read(GOOGLE_ZIP_BYTES, f"{d}/{fname}")
        try:
            values = parse_01_records(data)
            ok = len(values) == shots
            print(f"  {fname}: n_lines={len(values)} shots={shots} -> {'OK' if ok else 'MISMATCH'}")
        except ValueError as e:
            print(f"  {fname}: FAIL ({e})")


surface_code_bX_d3_r25_center_3_5
  obs_flips_actual.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_belief_matching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_correlated_matching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_pymatching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_tensor_network_contraction.01: n_lines=50000 shots=50000 -> OK
surface_code_bX_d3_r25_center_5_3
  obs_flips_actual.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_belief_matching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_correlated_matching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_pymatching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_tensor_network_contraction.01: n_lines=50000 shots=50000 -> OK
surface_code_bX_d3_r25_center_5_7
  obs_flips_actual.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicted_by_belief_matching.01: n_lines=50000 shots=50000 -> OK
  obs_flips_predicte

## Decoder mistakes and whether disagreement is useful

A mistake is `predicted != actual`, per shot per decoder. Also curious if the 4 decoders disagreeing with each other is a signal worth using.

In [9]:
DECODERS = ["pymatching", "correlated_matching", "belief_matching", "tensor_network_contraction"]

def load_01(d, fname):
    return parse_01_records(zip_read(GOOGLE_ZIP_BYTES, f"{d}/{fname}"))

mistake_rows = []
for d in experiment_dirs:
    actual = load_01(d, "obs_flips_actual.01")
    preds = {name: load_01(d, f"obs_flips_predicted_by_{name}.01") for name in DECODERS}

    for name, pred in preds.items():
        mistakes = sum(a != p for a, p in zip(actual, pred))
        mistake_rows.append(dict(experiment=d, decoder=name,
                                  mistakes=mistakes, rate=mistakes / len(actual)))

    disagree = sum(len({preds[n][i] for n in DECODERS}) > 1 for i in range(len(actual)))
    print(f"{d}: shots with any decoder disagreement = {disagree}/{len(actual)}")

    maj_correct = 0
    for i in range(len(actual)):
        votes = sum(preds[n][i] for n in DECODERS)
        maj = 1 if votes >= 3 else 0
        maj_correct += (maj == actual[i])
    best_single = 1 - min(r["rate"] for r in mistake_rows if r["experiment"] == d)
    print(f"{d}: majority-of-4 accuracy = {maj_correct/len(actual):.4f} "
          f"vs best single decoder = {best_single:.4f}")

decoder_summary = pd.DataFrame(mistake_rows)
decoder_summary.pivot(index="experiment", columns="decoder", values="rate")


surface_code_bX_d3_r25_center_3_5: shots with any decoder disagreement = 28429/50000
surface_code_bX_d3_r25_center_3_5: majority-of-4 accuracy = 0.5940 vs best single decoder = 0.5977
surface_code_bX_d3_r25_center_5_3: shots with any decoder disagreement = 27760/50000
surface_code_bX_d3_r25_center_5_3: majority-of-4 accuracy = 0.5820 vs best single decoder = 0.5870
surface_code_bX_d3_r25_center_5_7: shots with any decoder disagreement = 27609/50000
surface_code_bX_d3_r25_center_5_7: majority-of-4 accuracy = 0.6013 vs best single decoder = 0.6161
surface_code_bX_d3_r25_center_7_5: shots with any decoder disagreement = 27002/50000
surface_code_bX_d3_r25_center_7_5: majority-of-4 accuracy = 0.6049 vs best single decoder = 0.6116
surface_code_bX_d5_r25_center_5_5: shots with any decoder disagreement = 32372/50000
surface_code_bX_d5_r25_center_5_5: majority-of-4 accuracy = 0.5953 vs best single decoder = 0.6045


decoder,belief_matching,correlated_matching,pymatching,tensor_network_contraction
experiment,,,,
surface_code_bX_d3_r25_center_3_5,0.40250,0.41942,0.43416,0.40228
surface_code_bX_d3_r25_center_5_3,0.41592,0.42518,0.44318,0.41304
surface_code_bX_d3_r25_center_5_7,0.39196,0.41824,0.43064,0.38392
surface_code_bX_d3_r25_center_7_5,0.38836,0.41296,0.42994,0.38872
surface_code_bX_d5_r25_center_5_5,0.40148,0.42566,0.45102,0.39550


## d=3 vs d=5

Checking the raw detector/measurement/qubit counts differ between distances.

In [ ]:
for d in experiment_dirs:
    p = props_by_dir[d]
    print(d, "detectors=", p["circuit_detectors"],
          "measurements=", p["circuit_measurements"], "qubits=", p["circuit_qubits"])

## Dense vs sparse detector storage

How much smaller would this be if we only stored fired detectors?

In [11]:
d = experiment_dirs[0]
props = props_by_dir[d]
data = zip_read(GOOGLE_ZIP_BYTES, f"{d}/detection_events.b8")
records = list(iter_b8_records(data, bits_per_record=props["circuit_detectors"]))

dense_rows = props["shots"] * props["circuit_detectors"]
sparse_rows = sum(sum(r) for r in records)
print(f"{d}: dense rows={dense_rows:,}  sparse(fired=1) rows={sparse_rows:,}  "
      f"({sparse_rows/dense_rows:.1%} of dense)")


surface_code_bX_d3_r25_center_3_5: dense rows=10,000,000  sparse(fired=1) rows=1,491,971  (14.9% of dense)


## QASMBench files

For the 6 `.qasm` files (3 benchmarks, source + transpiled): file hash as a candidate stable id, register declarations, op counts (stripping gate *definitions* first so they don't get counted as executions), and any `if(syn==N) x q[i];` style corrections.

In [12]:
qasm_members = [n for n in zip_names(QASM_ZIP_BYTES) if n.endswith(".qasm")]
print(qasm_members)

REG_RE = re.compile(r"\b(qreg|creg)\s+(\w+)\[(\d+)\]")
GATE_DEF_RE = re.compile(r"gate\s+\w+[^{]*\{[^}]*\}", re.S)
IF_RE = re.compile(r"if\s*\(\s*(\w+)\s*==\s*(\d+)\s*\)\s*(\w+)\s+([\w\[\]]+)\s*;")

for member in qasm_members:
    raw = zip_read(QASM_ZIP_BYTES, member)
    file_hash = hashlib.sha256(raw).hexdigest()
    text = raw.decode()
    is_transpiled = "_transpiled" in member
    benchmark = member.rsplit("/", 1)[-1].replace(".qasm", "").replace("_transpiled", "")

    registers = REG_RE.findall(text)
    body_only = GATE_DEF_RE.sub("", text)  # strip gate DEFINITION bodies before counting ops

    cx_count = len(re.findall(r"\bcx\b", body_only))
    x_count = len(re.findall(r"\bx\b(?!\s*\()", body_only))
    measure_count = len(re.findall(r"\bmeasure\b", body_only))
    corrections = IF_RE.findall(body_only)  # (classical_reg, value, gate, target)

    print(member)
    print("  hash:", file_hash[:16], "...  transpiled:", is_transpiled, " benchmark:", benchmark)
    print("  registers:", registers)
    print("  op counts -> cx:", cx_count, "x:", x_count, "measure:", measure_count)
    print("  syndrome-controlled corrections (reg, value, gate, target):", corrections)


['small/error_correctiond3_n5/error_correctiond3_n5.qasm', 'small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm', 'small/qec_en_n5/qec_en_n5.qasm', 'small/qec_en_n5/qec_en_n5_transpiled.qasm', 'small/qec_sm_n5/qec_sm_n5.qasm', 'small/qec_sm_n5/qec_sm_n5_transpiled.qasm']
small/error_correctiond3_n5/error_correctiond3_n5.qasm
  hash: c33b5a03491bceb2 ...  transpiled: False  benchmark: error_correctiond3_n5
  registers: [('qreg', 'q', '5'), ('creg', 'c', '5')]
  op counts -> cx: 49 x: 0 measure: 5
  syndrome-controlled corrections (reg, value, gate, target): []
small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm
  hash: 10ca04e8c340ce46 ...  transpiled: True  benchmark: error_correctiond3_n5
  registers: [('qreg', 'q', '5'), ('creg', 'c', '5')]
  op counts -> cx: 49 x: 0 measure: 5
  syndrome-controlled corrections (reg, value, gate, target): []
small/qec_en_n5/qec_en_n5.qasm
  hash: d7b95967326b6505 ...  transpiled: False  benchmark: qec_en_n5
  registers: [(

## Data vs ancilla qubits

Parsing the custom `gate` defs and their call sites to bind formal params to actual qubit args, then splitting by prefix (`q[...]` = data, `a[...]` = ancilla) — this is roughly the parity-check-to-qubit mapping the `stabilizer_check` Silver table will need.

In [13]:
GATE_DEF_HEADER_RE = re.compile(r"gate\s+(\w+)\s+([\w,\s]+)\{(.*?)\}", re.S)
CALL_RE = re.compile(r"^\s*(\w+)\s+([\w\[\],\s]+);", re.M)

for member in qasm_members:
    if "_transpiled" in member:
        continue  # transpiled variants inline the gate body; use the source variant
    text = zip_read(QASM_ZIP_BYTES, member).decode()
    defs = {m[0]: (m[1], m[2]) for m in GATE_DEF_HEADER_RE.findall(text)}
    body = GATE_DEF_HEADER_RE.sub("", text)

    for call_gate, call_args in CALL_RE.findall(body):
        if call_gate in defs:
            formal_params, _def_body = defs[call_gate]
            actual_qubits = [a.strip() for a in call_args.split(",")]
            formal_names = [p.strip() for p in formal_params.split(",")]
            binding = dict(zip(formal_names, actual_qubits))
            print(member, "->", call_gate, "binding:", binding)
            data_participants = [q for q in actual_qubits if q.startswith("q[")]
            ancilla_participants = [q for q in actual_qubits if q.startswith("a[")]
            print("   data qubits:", data_participants, "ancillas:", ancilla_participants)


small/qec_sm_n5/qec_sm_n5.qasm -> syndrome binding: {'d1': 'q[0]', 'd2': 'q[1]', 'd3': 'q[2]', 'a1': 'a[0]', 'a2': 'a[1]'}
   data qubits: ['q[0]', 'q[1]', 'q[2]'] ancillas: ['a[0]', 'a[1]']
